In [4]:
import os
import sys 

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [6]:
spark = (
    SparkSession.builder
    .master('local')
    .appName('comerce')
    .getOrCreate()
)

## Arquivo sobre os produtos

In [7]:
df_produtos = spark.read.csv('C:/Users/cliente/Downloads/comerce/olist_products_dataset.csv',header=True,inferSchema=True)

In [11]:
df_produtos.limit(10).toPandas()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12


In [9]:
df_produtos = df_produtos.withColumnRenamed('product_id','ID_produto').withColumnRenamed('product_category_name','categoria_produto') \
.withColumnRenamed('product_name_lenght','tamanho_nome_produto').withColumnRenamed('product_description_lenght','tamanho_descricao_produto')\
.withColumnRenamed('product_photos_qty','qtd_fotos_produto').withColumnRenamed('product_weight_g','peso_produto')\
.withColumnRenamed('product_length_cm','tamanho_produto').withColumnRenamed('product_height_cm','altura_produto')\
.withColumnRenamed('product_width_cm','largura_produto')

In [10]:
df_produtos = df_produtos.drop(col('tamanho_nome_produto'))

In [142]:
df_produtos.limit(10).toPandas()

,ID_produto,categoria_produto,tamanho_descricao_produto,qtd_fotos_produto,peso_produto,tamanho_produto,altura_produto,largura_produto
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,402,4,625,20,17,13
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,745,1,200,38,5,11
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,1272,4,18350,70,24,44
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,184,2,900,40,8,40
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,163,1,400,27,13,17
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,1156,1,600,17,10,12


## ARQUIVO DE PAGAMENTOS

In [11]:
df_pagamentos = spark.read.csv('C:/Users/cliente/Downloads/comerce/olist_order_payments_dataset.csv',header=True,inferSchema=True)

In [146]:
df_pagamentos.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows



In [16]:
df_pagamentos = df_pagamentos.withColumnRenamed('order_id','ID_Pedido').withColumnRenamed('payment_sequential','sequencia_pagamento')\
.withColumnRenamed('payment_type','tipo_pagamento').withColumnRenamed('payment_installments','numero_parcelas_pagamento')\
.withColumnRenamed('payment_value','Valor_pagamento')

In [150]:
df_pagamentos.show(5)

+--------------------+-------------------+--------------+-------------------------+---------------+
|           ID_Pedido|sequencia_pagamento|tipo_pagamento|numero_parcelas_pagamento|Valor_pagamento|
+--------------------+-------------------+--------------+-------------------------+---------------+
|b81ef226f3fe1789b...|                  1|   credit_card|                        8|          99.33|
|a9810da82917af2d9...|                  1|   credit_card|                        1|          24.39|
|25e8ea4e93396b6fa...|                  1|   credit_card|                        1|          65.71|
|ba78997921bbcdc13...|                  1|   credit_card|                        8|         107.78|
|42fdf880ba16b47b5...|                  1|   credit_card|                        2|         128.45|
+--------------------+-------------------+--------------+-------------------------+---------------+
only showing top 5 rows



In [152]:
df_pagamentos.withColumn('Valor_pagamento',col('Valor_pagamento').cast(IntegerType())).printSchema()

root
 |-- ID_Pedido: string (nullable = true)
 |-- sequencia_pagamento: integer (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- numero_parcelas_pagamento: integer (nullable = true)
 |-- Valor_pagamento: integer (nullable = true)



In [154]:
df_pagamentos.select('tipo_pagamento').distinct().show(5)

+--------------+
|tipo_pagamento|
+--------------+
|        boleto|
|   not_defined|
|   credit_card|
|       voucher|
|    debit_card|
+--------------+



In [156]:
df_pagamentos.select('tipo_pagamento','valor_pagamento').orderBy(desc('valor_pagamento')).show(5)

+--------------+---------------+
|tipo_pagamento|valor_pagamento|
+--------------+---------------+
|   credit_card|       13664.08|
|        boleto|        7274.88|
|   credit_card|        6929.31|
|        boleto|        6922.21|
|        boleto|        6726.66|
+--------------+---------------+
only showing top 5 rows



In [18]:
df_pagamentos.createOrReplaceTempView("vw")
vw = spark.sql('''
SELECT 
CAST(valor_pagamento AS INT) AS valor_pagamento, * FROM vw
ORDER BY VALOR_PAGAMENTO DESC
''').printSchema()

root
 |-- valor_pagamento: integer (nullable = true)
 |-- ID_Pedido: string (nullable = true)
 |-- sequencia_pagamento: integer (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- numero_parcelas_pagamento: integer (nullable = true)
 |-- Valor_pagamento: double (nullable = true)



In [160]:
df_pagamentos.printSchema()

root
 |-- ID_Pedido: string (nullable = true)
 |-- sequencia_pagamento: integer (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- numero_parcelas_pagamento: integer (nullable = true)
 |-- Valor_pagamento: double (nullable = true)



In [162]:
df_pagamentos.groupBy(col('tipo_pagamento')).agg({'valor_pagamento':'avg'}).orderBy(desc(col('avg(valor_pagamento)'))).show(10)

+--------------+--------------------+
|tipo_pagamento|avg(valor_pagamento)|
+--------------+--------------------+
|   credit_card|  163.31902063935578|
|        boleto|   145.0344354023441|
|    debit_card|  142.57017004578162|
|       voucher|   65.70335411255424|
|   not_defined|                 0.0|
+--------------+--------------------+



In [125]:
df_pagamentos.count()

103886

## Arquivo de itens

In [20]:
df_itens = spark.read.csv('C:/Users/cliente/Downloads/comerce/olist_order_items_dataset.csv',header=True,inferSchema=True)

In [130]:
df_itens.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [22]:
df_itens = df_itens.withColumnRenamed('order_id','ID_Pedido').withColumnRenamed('order_item_id','ID_pedido_item')\
.withColumnRenamed('product_id','ID_produto').withColumnRenamed('seller_id','ID_venda')\
.withColumnRenamed('shipping_limit_date','data_limite_envio').withColumnRenamed('price','preco')\
.withColumnRenamed('freight_value','valor_frete')

In [24]:
df_join = df_pagamentos.join(df_itens, df_pagamentos.ID_Pedido == df_itens.ID_Pedido, 'inner')

In [176]:
df_join.limit(10).toPandas()

,ID_Pedido,sequencia_pagamento,tipo_pagamento,numero_parcelas_pagamento,Valor_pagamento,ID_Pedido,ID_pedido_item,ID_produto,ID_venda,data_limite_envio,preco,valor_frete
0,00010242fe8c5a6d1ba2dd792cb16214,1,credit_card,2,72.19,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,credit_card,3,259.83,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,credit_card,5,216.87,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,credit_card,2,25.78,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,credit_card,3,218.04,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,boleto,1,34.59,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,credit_card,1,31.75,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,credit_card,10,880.75,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,credit_card,3,157.60,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,credit_card,1,65.39,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


In [26]:
df_join_produto = df_join.join(df_produtos, df_join.ID_produto==df_produtos.ID_produto, 'inner')

In [190]:
df_join_produto.limit(10).toPandas()

,ID_Pedido,sequencia_pagamento,tipo_pagamento,numero_parcelas_pagamento,Valor_pagamento,ID_Pedido,ID_pedido_item,ID_produto,ID_venda,data_limite_envio,preco,valor_frete,ID_produto,categoria_produto,tamanho_descricao_produto,qtd_fotos_produto,peso_produto,tamanho_produto,altura_produto,largura_produto
0,00010242fe8c5a6d1ba2dd792cb16214,1,credit_card,2,72.19,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,598,4,650,28,9,14
1,00018f77f2f0320c557190d7a144bdd3,1,credit_card,3,259.83,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239,2,30000,50,30,40
2,000229ec398224ef6ca0657da4fc703e,1,credit_card,5,216.87,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,695,2,3050,33,13,33
3,00024acbcdf0a6daa1e931b038114c75,1,credit_card,2,25.78,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,7634da152a4610f1595efa32f14722fc,perfumaria,480,1,200,16,10,15
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,credit_card,3,218.04,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,409,1,3750,35,40,30
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,boleto,1,34.59,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,558,1,450,24,8,15
6,00054e8431b9d7675808bcb819fb4a32,1,credit_card,1,31.75,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia,815,1,200,27,5,20
7,000576fe39319847cbb9d288c5617fa6,1,credit_card,10,880.75,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,1310,3,13805,35,75,45
8,0005a1a1728c9d785b8e2b08b904576c,1,credit_card,3,157.60,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,493,1,2000,30,12,16
9,0005f50442cb953dcd1d21e1fb923495,1,credit_card,1,65.39,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos,1192,1,850,29,3,21


In [31]:
df_join_produto_2 = df_join_produto

In [33]:
df_join_produto_2.createOrReplaceTempView("TTS")

df_join_produto_2 = spark.sql('''
Select * from tts
WHERE AVG(
order by categoria_produto

SyntaxError: incomplete input (3858019105.py, line 3)

In [45]:
df_join_produto.groupBy('categoria_produto') \
    .agg(avg('Valor_pagamento').alias('total_vendas')) \
    .orderBy(desc('total_vendas')) \
    .show(10)

+--------------------+------------------+
|   categoria_produto|      total_vendas|
+--------------------+------------------+
|                 pcs| 1268.734318181819|
|      telefonia_fixa| 763.8754981549812|
|portateis_casa_fo...| 656.7861538461539|
|agro_industria_e_...|471.15321428571383|
|  eletrodomesticos_2| 464.7890298507464|
|   moveis_escritorio|363.79442632170986|
|sinalizacao_e_seg...| 340.7365671641791|
|construcao_ferram...|330.10567839195977|
|instrumentos_musi...| 324.6157660167132|
|     eletroportateis|321.80368045649055|
+--------------------+------------------+
only showing top 10 rows



In [47]:
df_join.groupBy('categoria_produto').orderBy(desc('preco')).show(5)

AttributeError: 'GroupedData' object has no attribute 'orderBy'

In [53]:

# Primeiro, realize o join entre df_join (pagamentos e itens) e df_produtos
#df_join_produto = df_join.join(df_produtos, 'ID_produto', 'inner')

# Agora você pode agrupar por categoria e encontrar o preço máximo
df_grouped = df_join_produto.groupBy('categoria_produto') \
    .agg(max('preco').alias('preco_maximo')) \
    .orderBy(desc('preco_maximo')) \
    .show(5)

+--------------------+------------+
|   categoria_produto|preco_maximo|
+--------------------+------------+
|utilidades_domest...|      6735.0|
|                 pcs|      6729.0|
|               artes|      6499.0|
|     eletroportateis|      4799.0|
|instrumentos_musi...|     4399.87|
+--------------------+------------+
only showing top 5 rows



In [57]:
df_join_produto.groupBy('categoria_produto').agg({'preco':'avg'}).show(10)

+--------------------+------------------+
|   categoria_produto|        avg(preco)|
+--------------------+------------------+
|                 pcs|1103.6891363636355|
|               bebes|135.94480913642218|
|               artes|112.88570776255706|
|           cine_foto| 95.14191780821912|
|    moveis_decoracao| 87.49930123513788|
|            pc_gamer|           167.594|
|construcao_ferram...|154.54770767613007|
|tablets_impressao...|  90.1185057471264|
|   artigos_de_festas|103.37347826086953|
|fashion_roupa_mas...| 80.21799999999985|
+--------------------+------------------+
only showing top 10 rows



In [59]:
df_join_produto.printSchema()

root
 |-- ID_Pedido: string (nullable = true)
 |-- sequencia_pagamento: integer (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- numero_parcelas_pagamento: integer (nullable = true)
 |-- Valor_pagamento: double (nullable = true)
 |-- ID_Pedido: string (nullable = true)
 |-- ID_pedido_item: integer (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- ID_venda: string (nullable = true)
 |-- data_limite_envio: timestamp (nullable = true)
 |-- preco: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- categoria_produto: string (nullable = true)
 |-- tamanho_descricao_produto: integer (nullable = true)
 |-- qtd_fotos_produto: integer (nullable = true)
 |-- peso_produto: integer (nullable = true)
 |-- tamanho_produto: integer (nullable = true)
 |-- altura_produto: integer (nullable = true)
 |-- largura_produto: integer (nullable = true)



In [67]:
df_join_produto = df_join_produto.withColumn('preco',col('preco').cast(IntegerType()))

In [75]:
df_join_produto.groupBy('categoria_produto').agg({'preco':'max'}).orderBy(desc('max(preco)')).show(10)

+--------------------+----------+
|   categoria_produto|max(preco)|
+--------------------+----------+
|utilidades_domest...|      6735|
|                 pcs|      6729|
|               artes|      6499|
|     eletroportateis|      4799|
|instrumentos_musi...|      4399|
|      consoles_games|      4099|
|       esporte_lazer|      4059|
|  relogios_presentes|      3999|
|                NULL|      3980|
|  ferramentas_jardim|      3930|
+--------------------+----------+
only showing top 10 rows



In [73]:
df_join_produto.show(5)

+--------------------+-------------------+--------------+-------------------------+---------------+--------------------+--------------+--------------------+--------------------+-------------------+-----+-----------+--------------------+------------------+-------------------------+-----------------+------------+---------------+--------------+---------------+
|           ID_Pedido|sequencia_pagamento|tipo_pagamento|numero_parcelas_pagamento|Valor_pagamento|           ID_Pedido|ID_pedido_item|          ID_produto|            ID_venda|  data_limite_envio|preco|valor_frete|          ID_produto| categoria_produto|tamanho_descricao_produto|qtd_fotos_produto|peso_produto|tamanho_produto|altura_produto|largura_produto|
+--------------------+-------------------+--------------+-------------------------+---------------+--------------------+--------------+--------------------+--------------------+-------------------+-----+-----------+--------------------+------------------+-------------------------

# Teste


In [78]:
df_join_produto.limit(10).toPandas()

,ID_Pedido,sequencia_pagamento,tipo_pagamento,numero_parcelas_pagamento,Valor_pagamento,ID_Pedido,ID_pedido_item,ID_produto,ID_venda,data_limite_envio,preco,valor_frete,ID_produto,categoria_produto,tamanho_descricao_produto,qtd_fotos_produto,peso_produto,tamanho_produto,altura_produto,largura_produto
0,00010242fe8c5a6d1ba2dd792cb16214,1,credit_card,2,72.19,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58,13.29,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,598,4,650,28,9,14
1,00018f77f2f0320c557190d7a144bdd3,1,credit_card,3,259.83,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239,19.93,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239,2,30000,50,30,40
2,000229ec398224ef6ca0657da4fc703e,1,credit_card,5,216.87,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199,17.87,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,695,2,3050,33,13,33
3,00024acbcdf0a6daa1e931b038114c75,1,credit_card,2,25.78,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12,12.79,7634da152a4610f1595efa32f14722fc,perfumaria,480,1,200,16,10,15
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,credit_card,3,218.04,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199,18.14,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,409,1,3750,35,40,30
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,boleto,1,34.59,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21,12.69,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,558,1,450,24,8,15
6,00054e8431b9d7675808bcb819fb4a32,1,credit_card,1,31.75,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19,11.85,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia,815,1,200,27,5,20
7,000576fe39319847cbb9d288c5617fa6,1,credit_card,10,880.75,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810,70.75,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,1310,3,13805,35,75,45
8,0005a1a1728c9d785b8e2b08b904576c,1,credit_card,3,157.60,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145,11.65,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,493,1,2000,30,12,16
9,0005f50442cb953dcd1d21e1fb923495,1,credit_card,1,65.39,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53,11.40,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos,1192,1,850,29,3,21


In [ ]:
df_join_produto.groupBy('ID_produto').agg(max('peso_produto')).orderBy(desc('max(peso_produto)')).show(5)

In [32]:
df_join_produto.withColumn('sequencia_pagamento', col('sequencia_pagamento').cast(DoubleType())).printSchema()


root
 |-- ID_Pedido: string (nullable = true)
 |-- sequencia_pagamento: double (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- numero_parcelas_pagamento: integer (nullable = true)
 |-- Valor_pagamento: double (nullable = true)
 |-- ID_Pedido: string (nullable = true)
 |-- ID_pedido_item: integer (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- ID_venda: string (nullable = true)
 |-- data_limite_envio: timestamp (nullable = true)
 |-- preco: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- categoria_produto: string (nullable = true)
 |-- tamanho_descricao_produto: integer (nullable = true)
 |-- qtd_fotos_produto: integer (nullable = true)
 |-- peso_produto: integer (nullable = true)
 |-- tamanho_produto: integer (nullable = true)
 |-- altura_produto: integer (nullable = true)
 |-- largura_produto: integer (nullable = true)



In [34]:
df_t = df_join_produto.withColumn('sequencia_pagamento', col('sequencia_pagamento').cast(DoubleType()))

In [36]:
df_t.printSchema()

root
 |-- ID_Pedido: string (nullable = true)
 |-- sequencia_pagamento: double (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- numero_parcelas_pagamento: integer (nullable = true)
 |-- Valor_pagamento: double (nullable = true)
 |-- ID_Pedido: string (nullable = true)
 |-- ID_pedido_item: integer (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- ID_venda: string (nullable = true)
 |-- data_limite_envio: timestamp (nullable = true)
 |-- preco: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- categoria_produto: string (nullable = true)
 |-- tamanho_descricao_produto: integer (nullable = true)
 |-- qtd_fotos_produto: integer (nullable = true)
 |-- peso_produto: integer (nullable = true)
 |-- tamanho_produto: integer (nullable = true)
 |-- altura_produto: integer (nullable = true)
 |-- largura_produto: integer (nullable = true)



In [38]:
df_t.withColumnRenamed('sequencia_pagamento', 'pagamento_tts').show(5)

+--------------------+-------------+--------------+-------------------------+---------------+--------------------+--------------+--------------------+--------------------+-------------------+-----+-----------+--------------------+------------------+-------------------------+-----------------+------------+---------------+--------------+---------------+
|           ID_Pedido|pagamento_tts|tipo_pagamento|numero_parcelas_pagamento|Valor_pagamento|           ID_Pedido|ID_pedido_item|          ID_produto|            ID_venda|  data_limite_envio|preco|valor_frete|          ID_produto| categoria_produto|tamanho_descricao_produto|qtd_fotos_produto|peso_produto|tamanho_produto|altura_produto|largura_produto|
+--------------------+-------------+--------------+-------------------------+---------------+--------------------+--------------+--------------------+--------------------+-------------------+-----+-----------+--------------------+------------------+-------------------------+-----------------

In [44]:
df_t.withColumn('tts', substring('tipo_pagamento', -4,4)).limit(5).toPandas()

,ID_Pedido,sequencia_pagamento,tipo_pagamento,numero_parcelas_pagamento,Valor_pagamento,ID_Pedido,ID_pedido_item,ID_produto,ID_venda,data_limite_envio,...,valor_frete,ID_produto,categoria_produto,tamanho_descricao_produto,qtd_fotos_produto,peso_produto,tamanho_produto,altura_produto,largura_produto,tts
0,00010242fe8c5a6d1ba2dd792cb16214,1.0,credit_card,2,72.19,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,...,13.29,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,598,4,650,28,9,14,card
1,00018f77f2f0320c557190d7a144bdd3,1.0,credit_card,3,259.83,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,...,19.93,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239,2,30000,50,30,40,card
2,000229ec398224ef6ca0657da4fc703e,1.0,credit_card,5,216.87,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,...,17.87,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,695,2,3050,33,13,33,card
3,00024acbcdf0a6daa1e931b038114c75,1.0,credit_card,2,25.78,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,...,12.79,7634da152a4610f1595efa32f14722fc,perfumaria,480,1,200,16,10,15,card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1.0,credit_card,3,218.04,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,...,18.14,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,409,1,3750,35,40,30,card


In [46]:
df_t.printSchema()

root
 |-- ID_Pedido: string (nullable = true)
 |-- sequencia_pagamento: double (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- numero_parcelas_pagamento: integer (nullable = true)
 |-- Valor_pagamento: double (nullable = true)
 |-- ID_Pedido: string (nullable = true)
 |-- ID_pedido_item: integer (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- ID_venda: string (nullable = true)
 |-- data_limite_envio: timestamp (nullable = true)
 |-- preco: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- ID_produto: string (nullable = true)
 |-- categoria_produto: string (nullable = true)
 |-- tamanho_descricao_produto: integer (nullable = true)
 |-- qtd_fotos_produto: integer (nullable = true)
 |-- peso_produto: integer (nullable = true)
 |-- tamanho_produto: integer (nullable = true)
 |-- altura_produto: integer (nullable = true)
 |-- largura_produto: integer (nullable = true)



In [64]:
df_t.withColumn('data_limite_envio', col('data_limite_envio').cast(DateType())).limit(5).toPandas()

,ID_Pedido,sequencia_pagamento,tipo_pagamento,numero_parcelas_pagamento,Valor_pagamento,ID_Pedido,ID_pedido_item,ID_produto,ID_venda,data_limite_envio,preco,valor_frete,ID_produto,categoria_produto,tamanho_descricao_produto,qtd_fotos_produto,peso_produto,tamanho_produto,altura_produto,largura_produto
0,00010242fe8c5a6d1ba2dd792cb16214,1.0,credit_card,2,72.19,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19,58.90,13.29,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,598,4,650,28,9,14
1,00018f77f2f0320c557190d7a144bdd3,1.0,credit_card,3,259.83,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03,239.90,19.93,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239,2,30000,50,30,40
2,000229ec398224ef6ca0657da4fc703e,1.0,credit_card,5,216.87,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18,199.00,17.87,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,695,2,3050,33,13,33
3,00024acbcdf0a6daa1e931b038114c75,1.0,credit_card,2,25.78,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15,12.99,12.79,7634da152a4610f1595efa32f14722fc,perfumaria,480,1,200,16,10,15
4,00042b26cf59d7ce69dfabb4e55b4fd9,1.0,credit_card,3,218.04,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13,199.90,18.14,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,409,1,3750,35,40,30
